# Colour-preference experiment — raw data preparation
This notebook creates the curated HDF5 files required by the Colour Preference analysis. Run it when newly recorded sessions do not yet contain curated output.

### Session 0.0: Locate shared code and import the extractor

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
# Imports required for raw LocustVR data preparation.
import os, json, sys
from pathlib import Path

current_working_directory = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in [current_working_directory, *current_working_directory.parents]
     if (candidate / 'locustvr_extractor.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Could not locate the UnityDataAnalysis repository root.')
workflow_dir = repo_root / 'analyses' / 'color_preference'
utilities_dir = repo_root.parent / 'utilities'
if not utilities_dir.is_dir():
    raise FileNotFoundError(f'Utilities directory not found: {utilities_dir}')
for import_dir in (repo_root, utilities_dir):
    if str(import_dir) not in sys.path:
        sys.path.insert(0, str(import_dir))

from locustvr_extractor import extract_locustvr_dat

#### Session 0.1: Load Colour Preference settings
Settings come from `analysis_methods_color_preference.json`. The current data root is the mounted AG Couzin-Fuchs LocustVR directory.

In [ ]:
json_file = workflow_dir / 'analysis_methods_color_preference.json'
with open(json_file, "r") as f:
    analysis_methods = json.loads(f.read())

thisDataset = "/Volumes/AG_Couzin-Fuchs/DATA/experiment_trackball_Optomotor/locustVR"
exp_name = analysis_methods.get("experiment_name")
if exp_name != "locustvr":
    raise ValueError(f"This branch only supports LocustVR data, not {exp_name!r}.")

#### Session 0.2: Discover raw recording sessions
Every folder containing a `.dat` file is added to `dir_list`. Inspect the list before starting extraction, or activate the optional manual selection for a smaller batch.

In [ ]:
# Find all session folders containing raw LocustVR tracking files.
dir_list = []
for root, dirs, files in os.walk(thisDataset):
    for folder in dirs:
        folder_path=os.path.join(root,folder)
        if any(name.endswith(".dat") for name in os.listdir(folder_path)):
            dir_list.append(folder_path.replace("\\", "/"))

dir_list.sort()
print(f"{dir_list}", len(dir_list))

In [ ]:
# Optional manual selection. Leave disabled to analyse every discovered session.
# dir_list = dir_list[-8:]

#### Session 0.3: Optionally copy temperature information
This step is disabled by default. Enable it only when the specified logger file belongs to the recordings being processed.

In [ ]:
## Optional: copy one thermo-humidity log into every selected data folder.
## Keep this disabled unless tmp_source points to the logger file for this experiment.
copy_temperature_file = False
if copy_temperature_file:
    import shutil
    tmp_file_name = 'locustVR_20251127_20251204.txt'
    tmp_source = os.path.join('/Volumes/AG_Couzin-Fuchs/Users/chiyu', tmp_file_name)
    if not os.path.isfile(tmp_source):
        raise FileNotFoundError(f'Temperature file not found: {tmp_source}')
    for this_dir in dir_list:
        tmp_new_dir = os.path.join(this_dir, tmp_file_name)
        if os.path.isfile(tmp_new_dir):
            print('Found temperature file in the data folder already')
        else:
            shutil.copy(tmp_source, tmp_new_dir)

In [ ]:
# Optional manual selection. Leave disabled for the current LocustVR dataset.
# dir_list = dir_list[42:]

### Session 1.0: Create curated Colour Preference HDF5 files
For each selected session, `extract_locustvr_dat` creates the tracking and stimulus HDF5 files. With `overwrite_curated_dataset = false`, sessions containing curated HDF5 output are skipped.

In [ ]:
# Create the curated tracking and stimulus HDF5 files for each session.
pattern = "*.h5"
for this_dir in dir_list:
    if "archive" in this_dir:
        print(f"skip archive folder for {this_dir}")
        continue
    if any(Path(this_dir).glob(pattern)) and not analysis_methods.get("overwrite_curated_dataset", False):
        print(f"curated LocustVR h5 database found in {this_dir}. Skip this file")
        continue
    print(f"no curated LocustVR h5 database in {this_dir}. Create curated files")
    extract_locustvr_dat(this_dir, analysis_methods)